# LLM Providers and the ReAct Loop

**Docker image**: `ml4t`

A **forecasting agent** answers a question by choosing what to look up, reading what comes
back, and then committing to a number. That is a different shape of program from the models
in earlier chapters, which are handed a prepared feature matrix and produce a prediction from
it. This notebook builds the smallest useful version: a loop that alternates between
reasoning about what it still needs and acting to get it, backed by a language model that can
be swapped without touching the loop.

**Learning Objectives**:
- Call any of five language-model backends through one two-method interface, so the same
  agent code runs against a free local model or a commercial API
- Write a loop that alternates reasoning and tool calls (the **ReAct** pattern) and stops
  either on a forecast or on a step budget you set
- Validate a model's reply against a declared action schema before acting on it, and return
  a malformed reply to the model for correction
- Read a saved run record to see which searches were issued, what came back, and what the
  loop did with each result

**Book Reference**: Chapter 24, Sections 24.1-24.2 (From Prediction Functions to Agentic
Workflows; Cognitive Architectures)

**Prerequisites**: None. This is the first notebook in the Chapter 24 workshop.

<!-- ML4T_ZH_BEGIN c0000-p000 -->
# LLM 供應者與 ReAct 迴圈

**Docker 映像檔**：`ml4t`

**預測代理**會選擇要查詢的資訊、閱讀查詢結果，再給出數值，以回答一個問題。這種程式的運作方式不同於前幾章的模型：那些模型接收事先備妥的特徵矩陣，並據此產生預測。本 Notebook 建立一個最精簡而實用的版本：在推理尚需哪些資訊與採取行動取得資訊之間交替的迴圈，背後使用可替換的語言模型，替換時無須修改迴圈。

**學習目標**：
- 透過統一的雙方法介面呼叫五種語言模型後端，讓同一份代理程式碼能搭配免費的本機模型或商業 API 執行
- 撰寫在推理與工具呼叫之間交替的迴圈（**ReAct** 模式），並在產生預測或達到所設步數上限時停止
- 依宣告的行動綱要驗證模型回覆，再據以採取行動；若回覆格式錯誤，則送回模型修正
- 閱讀儲存的執行紀錄，了解發出了哪些搜尋、得到什麼結果，以及迴圈如何處理每項結果

**書籍參考**：第 24 章，第 24.1–24.2 節（從預測函數到代理式工作流程；認知架構）

**先備知識**：無。本 Notebook 是第 24 章實作課程的第一份。
<!-- ML4T_ZH_END c0000-p000 -->


In [2]:
"""LLM Providers and the ReAct Loop - multi-provider agent reasoning."""

import sys

from utils.paths import get_chapter_dir

sys.path.insert(0, str(get_chapter_dir(24)))

import json
from datetime import date

from agent_fixtures import get_live_question
from agent_observability import TRACES_DIR, RunTrace, trace_llm
from agent_pipeline import validate_probabilities
from agent_providers import ChatMessage, MockLLMClient, TokenUsage, create_llm_client
from agent_schemas import AgentForecastArtifact, AgentTrace, ForecastQuestion
from agent_tools import ToolExecutor, create_search_client, format_search_results

In [3]:
# RUN_LIVE=False (the default) replays a pinned claude-sonnet + Tavily run: the
# notebook reloads the saved trace named below, makes no API calls, and
# reproduces the captured ReAct session, so the outputs are stable and match the
# narrative. Set RUN_LIVE=True (with LLM_PROVIDER + LLM_API_KEY + TAVILY_API_KEY in .env) to
# fetch a fresh live question and run the agent against it; that path makes real
# calls and is not reproducible.
RUN_LIVE = False
PINNED_TRACE = "01_react_reasoning_20260615T191047Z_04eb6e7c603d.json"

LLM_PROVIDER = ""  # Blank reads .env; "mock" selects a synthetic fixture
MAX_STEPS = 6

## The LLM Provider Protocol

Every notebook in this chapter reaches its language model through the same `LLMClient`
**protocol**: a pair of method signatures that any provider class must offer, with no shared
base class and no vendor SDK visible to the caller.

- `complete(messages) → str` returns the reply text
- `complete_with_usage(messages) → (str, TokenUsage)` returns the text and the prompt and
  completion token counts, which is what makes a run's cost measurable

Two methods is a deliberate floor. Everything a provider offers beyond them, such as
streaming, native tool calling or thinking budgets, varies across vendors, and an agent that
reaches for any of it is an agent that can no longer be moved.

The cell below sends one arithmetic question. Its answer is worthless; what it establishes is
that a client was constructed, a key was accepted, and a reply came back, before the loop
starts spending money on real prompts. On the replay path there is no client, so the reply
comes out of the saved record.

<!-- ML4T_ZH_BEGIN c0003-p000 -->
## LLM 供應者協定

本章每份 Notebook 都透過相同的 `LLMClient` **協定**連接語言模型：任何供應者類別都必須提供一對方法簽章，無須共用基底類別，呼叫端也看不到供應商 SDK。

- `complete(messages) → str` 回傳回覆文字
- `complete_with_usage(messages) → (str, TokenUsage)` 回傳文字、提示詞及回覆的 token 數量，讓執行成本可以衡量

只要求兩個方法是刻意設定的最低門檻。供應者在此之外提供的串流、原生工具呼叫或思考預算等功能，各家都不同；代理一旦使用其中任何功能，就無法再自由移轉。

下方儲存格送出一道算術題。答案本身沒有價值；這個測試是為了在迴圈開始為實際提示詞花錢之前，確認用戶端已建立、金鑰可用，而且收到回覆。若使用重播路徑，則不建立用戶端，回覆來自已儲存的紀錄。
<!-- ML4T_ZH_END c0003-p000 -->


**Optional dependencies** (for a real LLM and web search - the chapter runs in
offline replay mode without them):

```bash
uv pip install anthropic openai httpx        # cloud providers
# Ollama (free, local) - install from https://ollama.com, then:
#   ollama serve && ollama pull qwen3:8b
```

Then set your provider in the repo-root `.env` (copy it first:
`cp .env.example .env`). You set three things - **never a base URL**:

- `LLM_PROVIDER` - `deepseek` | `openrouter` | `openai` | `anthropic` | `google` | `ollama` | `mock`
- `LLM_API_KEY` - your key for that provider
- `LLM_MODEL` - *optional*; blank uses the provider default
- `TAVILY_API_KEY` - web search ([tavily.com](https://tavily.com))

Live mode auto-loads `.env`. Missing live credentials raise an explicit error;
select `mock` only for a labeled synthetic test fixture. Run `uv run python check_env.py` to confirm which
model will be used.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
**選用相依套件**（用於實際 LLM 與網頁搜尋；沒有這些套件，本章仍可在離線重播模式執行）：

```bash
uv pip install anthropic openai httpx        # cloud providers
# Ollama (free, local) - install from https://ollama.com, then:
#   ollama serve && ollama pull qwen3:8b
```

接著在儲存庫根目錄的 `.env` 設定供應者（先複製：`cp .env.example .env`）。需要設定三項，**切勿設定 base URL**：

- `LLM_PROVIDER`：`deepseek` | `openrouter` | `openai` | `anthropic` | `google` | `ollama` | `mock`
- `LLM_API_KEY`：該供應者的金鑰
- `LLM_MODEL`：*選填*；留空則使用供應者的預設值
- `TAVILY_API_KEY`：網頁搜尋（[tavily.com](https://tavily.com)）

即時模式會自動載入 `.env`。缺少即時模式憑證時會明確報錯；只有在使用有標示的合成測試資料時才選擇 `mock`。執行 `uv run python check_env.py`，確認將使用哪個模型。
<!-- ML4T_ZH_END c0004-p000 -->


In [4]:
if RUN_LIVE:
    llm = create_llm_client(LLM_PROVIDER)
    provider_name = llm.model_name
    response = llm.complete(
        [ChatMessage(role="user", content="What is 2 + 2? Reply with just the number.")]
    )
else:
    # Replay: reload the pinned trace; the provider name and warm-up response
    # come from the saved run, and no live client is created.
    pinned_run = RunTrace.load(TRACES_DIR / PINNED_TRACE)
    llm = None
    provider_name = pinned_run.provider
    response = pinned_run.params.get("warmup_response", "")

print(f"Mode:     {'LIVE' if RUN_LIVE else 'REPLAY (pinned trace)'}")
print(f"Provider: {provider_name}")
print(f"Response: {response[:200]}")

Mode:     REPLAY (pinned trace)
Provider: claude-sonnet-4-6
Response: 4


## The Forecasting Question

The agent is pointed at a **prediction market** question: a contract that pays out if a
stated event happens by a stated date, so its price is the market's probability that it will.
Polymarket publishes these openly, and an unresolved one is useful here for a reason that
matters when evaluating any language model on forecasting. A question whose answer is already
in the model's training data tests recall, not forecasting. An open question does not exist in
the training data at all, so the only way to say anything about it is to go and look.

On the replay path the question comes out of the saved record. On the live path
`get_live_question` calls the Polymarket API and falls back to a fixed demonstration question
when that call fails, so the notebook still runs with no network.

<!-- ML4T_ZH_BEGIN c0006-p000 -->
## 預測問題

代理要處理一個**預測市場**問題：若指定事件在指定日期前發生，合約就會支付款項，因此合約價格代表市場認為事件會發生的機率。Polymarket 公開發布這些合約，而尚未結算的問題對評估語言模型預測能力特別有用。如果問題的答案已存在模型訓練資料中，測到的是記憶，而非預測。未解決問題的答案根本不在訓練資料裡，因此要對它提出看法，唯一途徑就是查找資訊。

在重播路徑中，問題取自已儲存的紀錄。在即時路徑中，`get_live_question` 會呼叫 Polymarket API；若呼叫失敗，便改用固定的示範問題，讓 Notebook 在沒有網路時仍可執行。
<!-- ML4T_ZH_END c0006-p000 -->


In [5]:
question = get_live_question() if RUN_LIVE else pinned_run.question_obj()
print(f"Question: {question.question}")
if question.resolution_date:
    print(f"Resolves: {question.resolution_date}")
if question.current_market_price is not None:
    print(f"Market:   {question.current_market_price:.0%}")
if question.resolved_outcome is not None:
    print(f"Outcome:  {'YES' if question.resolved_outcome == 1.0 else 'NO'}")
else:
    print("Status:   OPEN (unresolved)")

Question: Will no Fed rate cuts happen in 2026?
Resolves: 2026-12-31
Market:   71%
Status:   OPEN (unresolved)


## Building the ReAct Agent

The ReAct pattern interleaves **reasoning** (thinking about what to do) with **action**
(calling tools) in a loop. The agent here has exactly two actions:

- `{"action": "search", "query": "..."}`: search the web for evidence
- `{"action": "forecast", "p_yes": 0.XX, "rationale": "..."}`: produce a probability

The prompts appear in full below rather than hidden behind a helper, because what the model
is told is as much a part of this program as the loop that calls it. The same two-action
schema is used by the AIA Forecaster (Alur et al., 2025), the system the rest of this chapter
reconstructs.

<!-- ML4T_ZH_BEGIN c0008-p000 -->
## 建立 ReAct 代理

ReAct 模式在迴圈中交替進行**推理**（思考接下來該做什麼）與**行動**（呼叫工具）。這裡的代理只有兩種行動：

- `{"action": "search", "query": "..."}`：搜尋網路上的證據
- `{"action": "forecast", "p_yes": 0.XX, "rationale": "..."}`：給出機率

提示詞完整列在下方，而非藏在輔助函式中，因為模型收到的指示和呼叫模型的迴圈一樣，都是程式的一部分。相同的雙行動綱要也用於 AIA Forecaster（Alur 等人，2025），本章後續會重建該系統。
<!-- ML4T_ZH_END c0008-p000 -->


In [6]:
SYSTEM_PROMPT = """\
You are a forecasting agent in a multi-agent forecasting system.

Your job:
1) Gather evidence by issuing web/news search queries when needed.
2) Then produce a binary probability forecast for the question.

You must follow the action schema exactly and output valid JSON only.
You must not browse prediction market prices unless they are explicitly provided."""
SYSTEM_PROMPT += (
    "\nCite a retained source URL or title in the rationale. Abstain if evidence is insufficient."
)

### Step prompt

The step prompt provides the question, optional market context, and the action
schema. The agent must output exactly one JSON action per step.

<!-- ML4T_ZH_BEGIN c0010-p000 -->
### 步驟提示詞

步驟提示詞提供問題、選用的市場脈絡及行動綱要。代理每一步都必須只輸出一個 JSON 行動。
<!-- ML4T_ZH_END c0010-p000 -->


In [7]:
def build_step_prompt(q: ForecastQuestion, market_price: float | None = None) -> str:
    """Format the step prompt with question context."""
    prompt = f"QUESTION:\n{q.question}\n\n"
    if q.description:
        prompt += f"MARKET CONTEXT:\n{q.description}\n\n"
    if market_price is not None:
        prompt += f"MARKET IMPLIED PROBABILITY (p_yes):\n{market_price}\n\n"
    prompt += (
        "SEARCH BUDGET: search only when a specific fact is missing; do not "
        "reformulate a query you already ran. Once you have a base rate and the "
        "current signal, forecast.\n\n"
        "NEXT ACTION SCHEMA (output JSON only):\n"
        'If you need more info:\n{"action":"search","query":"..."}\n'
        "If you are ready to forecast:\n"
        '{"action":"forecast","p_yes":0.XX,"rationale":"short explanation '
        'grounded in evidence and base rates"}\n\n'
        'If evidence is insufficient: {"action":"abstain","rationale":"why"}\n'
        "Pick exactly one action."
    )
    return prompt

### Session setup

A session opens with three pieces of state. The **cutoff date** is the last date the search
tool may return documents from, which is what keeps an agent forecasting a past question from
reading the answer. The tool executor holds the search client and is the only route the loop
has to the outside world. The message list starts with the system prompt and the first step
prompt, and grows by two entries per turn: what the model said, and what the tool returned.

<!-- ML4T_ZH_BEGIN c0012-p000 -->
### 工作階段設定

工作階段開始時有三項狀態。**截止日期**是搜尋工具可回傳文件的最晚日期，藉此防止代理在回測過去的問題時讀到答案。工具執行器持有搜尋用戶端，也是迴圈連接外部世界的唯一途徑。訊息清單一開始包含系統提示詞與第一個步驟提示詞，之後每輪增加兩筆：模型所說的內容，以及工具回傳的內容。
<!-- ML4T_ZH_END c0012-p000 -->


In [8]:
def _init_react_session(
    question: ForecastQuestion,
    search_client,
    market_price: float | None,
) -> tuple[date | None, ToolExecutor, list[ChatMessage]]:
    """Build the cutoff date, tool executor, and initial messages for a ReAct run."""
    cutoff = date.fromisoformat(question.cutoff_date) if question.cutoff_date else None
    executor = ToolExecutor(search=search_client)
    messages = [
        ChatMessage(role="system", content=SYSTEM_PROMPT),
        ChatMessage(role="user", content=build_step_prompt(question, market_price)),
    ]
    return cutoff, executor, messages

### The ReAct loop

Each iteration: the LLM decides what to do (search or forecast), we
execute it, and feed the observation back. The loop terminates when the
agent issues a `forecast` action or hits the step limit. The `p_yes`
value must be a finite number in [0, 1], with a rationale identifying
retained evidence. Invalid or unsupported output is rejected. The budget
can end with abstention rather than a probability.

<!-- ML4T_ZH_BEGIN c0014-p000 -->
### ReAct 迴圈

每次迭代時，LLM 決定要做什麼（搜尋或預測），我們執行該行動，並將觀察結果回饋給它。代理發出 `forecast` 行動或達到步數上限時，迴圈就會結束。`p_yes` 值必須是 [0, 1] 範圍內的有限數值，並附上指出保留證據的理由。無效或缺乏依據的輸出會遭拒絕。用盡預算時，代理可以棄答，而不一定要給出機率。
<!-- ML4T_ZH_END c0014-p000 -->


In [9]:
def run_react_agent(
    llm,
    search_client,
    question: ForecastQuestion,
    max_steps: int = 5,
    market_price: float | None = None,
) -> tuple[float | None, str, list[AgentTrace], TokenUsage]:
    if max_steps < 1:
        raise ValueError("max_steps must be positive")
    cutoff, executor, messages = _init_react_session(question, search_client, market_price)
    traces, sources = [], []
    total_tokens = TokenUsage()
    reason = "Budget exhausted without a supported forecast"
    for step in range(max_steps + 1):
        synthesis_only = step == max_steps
        if synthesis_only:
            if not sources:
                break
            messages.append(
                ChatMessage(
                    role="user",
                    content="No more searches. Cite retained evidence and forecast, or abstain.",
                )
            )
        try:
            raw, usage = llm.complete_with_usage(messages, json_mode=True)
        except Exception as exc:
            reason = f"Model call failed ({type(exc).__name__})"
            traces.append(AgentTrace(step=step, action="provider_error", llm_raw=reason))
            break
        total_tokens += usage
        messages.append(ChatMessage(role="assistant", content=raw))
        try:
            action = json.loads(raw)
            if not isinstance(action, dict):
                raise ValueError("Action must be an object")
        except (ValueError, TypeError):
            action = {}
        kind = action.get("action", "invalid")
        if kind == "search" and not synthesis_only:
            query = action.get("query", "")
            results = executor.execute_search(query, max_results=5, cutoff_date=cutoff)
            sources.extend(results)
            traces.append(
                AgentTrace(step=step, action="search", query=query, results=results, llm_raw=raw)
            )
            messages.append(ChatMessage(role="tool", content=format_search_results(results)))
            print(f"Step {step}: search -> {len(results)} eligible results")
            continue
        traces.append(AgentTrace(step=step, action=str(kind), llm_raw=raw))
        if kind == "abstain":
            reason = str(action.get("rationale") or "Insufficient evidence")
            break
        try:
            if kind != "forecast":
                raise ValueError("Invalid action or exhausted search budget")
            if isinstance(action["p_yes"], bool):
                raise ValueError("Probability must be a number, not a boolean")
            probability = float(action["p_yes"])
            validate_probabilities([probability])
            rationale = action.get("rationale")
            if not isinstance(rationale, str) or not any(
                (s.url and s.url in rationale) or (s.title and s.title in rationale)
                for s in sources
            ):
                raise ValueError("Forecast needs a rationale identifying retained evidence")
        except (KeyError, TypeError, ValueError) as exc:
            reason = str(exc)
            messages.append(ChatMessage(role="tool", content=reason))
            continue
        return probability, rationale, traces, total_tokens
    return None, reason, traces, total_tokens

## Running the Agent

The agent chooses each turn: search again, or commit to a probability. What follows is the
recorded session, one line per turn.

On the default replay path the run does not reach a forecast. The model kept searching for
evidence that the 2026 rate path had already been settled, found reporting that argued both
ways, and used every turn doing so. That is the ordinary failure of an agent under a budget,
and it is why `run_react_agent` returns an explicit no-answer value instead of a probability:
a caller that reads that value as a forecast would record a confident coin-flip where the
agent said nothing. The mock run at the end of the notebook takes the other branch and
commits.

<!-- ML4T_ZH_BEGIN c0016-p000 -->
## 執行代理

每一輪，代理都會選擇繼續搜尋，或決定給出一個機率。以下是記錄的工作階段，每輪一行。

在預設的重播路徑中，這次執行並未得出預測。模型持續搜尋證據，想確認 2026 年利率走勢是否已經確定；它找到支持不同方向的報導，並把所有輪次都花在搜尋上。這是代理在預算限制下常見的失敗情況，也正是 `run_react_agent` 會明確回傳「無答案」值，而非機率的原因：若呼叫端把該值當成預測，就會在代理根本沒有提出判斷時，記錄一個看似有把握的五五波預測。Notebook 結尾的模擬執行走另一條路徑，會給出確定的預測。
<!-- ML4T_ZH_END c0016-p000 -->


In [10]:
if RUN_LIVE:
    search_client = create_search_client()
    search_name = type(search_client).__name__
    # Wrap the client so every prompt/response is captured for the run trace.
    tracer = trace_llm(llm, label="react_agent")
    print(f"Search: {search_name}\n")
    print(f"Question: {question.question}\n")
    p_yes, rationale, traces, tokens = run_react_agent(
        tracer, search_client, question, max_steps=MAX_STEPS
    )
else:
    # Replay: rehydrate the pinned agent and reconstruct the step-by-step log
    # that run_react_agent prints during a live run.
    artifact = pinned_run.agent_artifacts()[0]
    p_yes, rationale, traces, tokens = (
        artifact.p_yes,
        artifact.rationale,
        artifact.traces,
        artifact.token_usage,
    )
    if not any(t.action == "forecast" for t in traces):
        p_yes = None
        rationale = "No forecast action in this capture; the saved 0.5 is a legacy sentinel"
    search_name = pinned_run.params.get("search_client", "replay (pinned trace)")
    print(f"Search: {search_name}\n")
    print(f"Question: {question.question}\n")
    for t in traces:
        if t.action == "search":
            print(f'  Step {t.step}: search("{t.query}") → {len(t.results)} results')
        elif t.action == "forecast":
            print(f"  Step {t.step}: forecast → p_yes={p_yes:.2f}")

Search: TavilySearchClient



Question: Will no Fed rate cuts happen in 2026?

  Step 1: search("Fed rate cuts 2026 expectations FOMC forecast") → 5 results
  Step 2: search("Fed rate cuts 2026 current status how many cuts so far June 2026") → 5 results
  Step 3: search("Fed rate cuts 2026 zero cuts hold rates inflation Reuters June 2026") → 5 results
  Step 4: search("Fed rate cut 2026 any cuts made so far current federal funds rate") → 5 results
  Step 5: search("Fed rate cut 2026 probability rest of year outlook inflation hike") → 5 results


In [11]:
print("--- Forecast ---" if p_yes is not None else "--- Abstention ---")
print(f"p(YES) = {p_yes}")
print(f"Rationale: {rationale}")
print(f"Tokens: {tokens.total_tokens:,}")

--- Abstention ---
p(YES) = None
Rationale: No forecast action in this capture; the saved 0.5 is a legacy sentinel
Tokens: 25,696


## Inspecting the Execution Trace

Every step is captured as an `AgentTrace`. This is critical for **auditability**:
in production, you need to know which queries were issued, what results came back,
and how the LLM arrived at its forecast.

<!-- ML4T_ZH_BEGIN c0019-p000 -->
## 檢查執行軌跡

每個步驟都會記錄為 `AgentTrace`。這對**可稽核性**至關重要：在正式環境中，必須知道發出了哪些查詢、取得什麼結果，以及 LLM 如何得出預測。
<!-- ML4T_ZH_END c0019-p000 -->


In [12]:
print(f"Total steps: {len(traces)}")
print(f"Token usage: {tokens.total_tokens:,} tokens\n")

for t in traces:
    if t.action == "search":
        print(f'Step {t.step} [SEARCH] query="{t.query}"')
        for r in t.results[:3]:
            print(f"  → {r.title[:70]}")
    elif t.action == "forecast":
        data = json.loads(t.llm_raw) if t.llm_raw else {}
        print(f"Step {t.step} [FORECAST] p_yes={data.get('p_yes', '?')}")
    print()

Total steps: 5
Token usage: 25,696 tokens

Step 1 [SEARCH] query="Fed rate cuts 2026 expectations FOMC forecast"
  → What's The Fed's Next Move? | J.P. Morgan Global Research
  → Fed Outlook 2026: Rate Forecasts and Fixed Income Strategies
  → Fed predictions for 2026: What experts say about the possibility of ..

Step 2 [SEARCH] query="Fed rate cuts 2026 current status how many cuts so far June 2026"
  → Will US Interest Rates Fall More in 2026? Our Latest Forecast
  → The Fed Just Cut Rates Again. Here's What's Ahead for 2026
  → Fed to hold rates this year, cut calls fade as war inflation ... - Reu

Step 3 [SEARCH] query="Fed rate cuts 2026 zero cuts hold rates inflation Reuters June 2026"
  → Fed to hold rates through 2026 amid persistent inflation: Reuters
  → Nomura pivots away from Fed rate cuts in 2026 as inflation risks linge
  → Fed rate cut pushed back to late 2026 on war-related inflation risks: 

Step 4 [SEARCH] query="Fed rate cut 2026 any cuts made so far current federal

## Persisting the Run

A live run depends on three things that will not exist tomorrow: an open market question, the
documents a search API returned that day, and a specific model version. Writing the question,
the agent's steps, and the full prompt-and-reply conversation into one JSON file under
`forecast_traces/` fixes all three. That file is what `RUN_LIVE = False` reads, which is why
the printed session above is the same for every reader regardless of which provider or search
backend they have configured, and why a run can be re-examined months later.

<!-- ML4T_ZH_BEGIN c0021-p000 -->
## 保存執行紀錄

即時執行仰賴三項明天可能就不存在的事物：尚未結算的市場問題、搜尋 API 當天回傳的文件，以及特定的模型版本。將問題、代理步驟和完整的提示詞與回覆對話寫入 `forecast_traces/` 下的單一 JSON 檔案，就能將三者固定下來。`RUN_LIVE = False` 讀取的正是這個檔案，因此無論讀者設定哪一家模型供應者或搜尋後端，上方印出的工作階段都一樣；數個月後也仍可重新檢視這次執行。
<!-- ML4T_ZH_END c0021-p000 -->


In [13]:
if RUN_LIVE:
    artifact = AgentForecastArtifact(
        agent_id="react_agent",
        p_yes=p_yes,
        rationale=rationale,
        traces=traces,
        token_usage=tokens,
        status="accepted" if p_yes is not None else "abstained",
        failure_reason=rationale if p_yes is None else None,
        execution_mode="synthetic" if "mock" in llm.model_name else "live",
        search_queries_made=sum(1 for t in traces if t.action == "search"),
        sources_consulted=sum(len(t.results) for t in traces),
    )
    run = RunTrace.capture(
        notebook="01_react_reasoning",
        provider=provider_name,
        question=question,
        params={
            "max_steps": MAX_STEPS,
            "warmup_response": response,
            "search_client": search_name,
        },
        agents=[artifact],
        llm_calls=tracer.calls,
        notes="ReAct agent on a live prediction-market question.",
    )
    trace_path = run.save()
    print(
        f"Saved {len(run.llm_calls)} model calls ({run.total_tokens():,} tokens) → {trace_path.name}"
    )
else:
    print(
        f"Replayed {len(pinned_run.llm_calls)} model calls "
        f"({pinned_run.total_tokens():,} tokens) from {PINNED_TRACE}"
    )

Replayed 5 model calls (25,696 tokens) from 01_react_reasoning_20260615T191047Z_04eb6e7c603d.json


## Swapping the Backend

Nothing in `run_react_agent` names a provider. It calls `complete_with_usage` on whatever
object it was handed, so any class satisfying the two-method protocol can be substituted:
`MockLLMClient` here, a local Ollama model, or a commercial API. The cell below runs the same
loop against the mock client, whose replies are canned, and gives it three turns rather than
five because the mock searches once and then commits.

On the live path, `create_llm_client("")` picks the first provider whose key is present, and
reading `LLM_PROVIDER` from the environment overrides that. So a continuous-integration run
forces deterministic replies without editing the notebook:

```bash
LLM_PROVIDER=mock uv run python 24_autonomous_agents/01_react_reasoning.py
```

That variable only takes effect when `RUN_LIVE = True`; on the replay path no client is
constructed at all.

<!-- ML4T_ZH_BEGIN c0023-p000 -->
## 替換後端

`run_react_agent` 中沒有指定任何供應者。它對傳入的物件呼叫 `complete_with_usage`，因此任何符合雙方法協定的類別都能替換進來：此處的 `MockLLMClient`、本機 Ollama 模型或商業 API。下方儲存格以回覆內容預先設定的模擬用戶端執行相同迴圈；由於模擬流程只搜尋一次就給出預測，因此給它三輪，而非五輪。

在即時路徑中，`create_llm_client("")` 會選擇第一個有金鑰的供應者；從環境讀取 `LLM_PROVIDER` 可覆寫這個選擇。因此，持續整合執行時可以強制使用可重現的回覆，而無須編輯 Notebook：

```bash
LLM_PROVIDER=mock uv run python 24_autonomous_agents/01_react_reasoning.py
```

該變數只有在 `RUN_LIVE = True` 時才會生效；重播路徑完全不建立用戶端。
<!-- ML4T_ZH_END c0023-p000 -->


The June capture retains actual searches and model messages. It predates the new
acceptance check; inspect its rationale and sources rather than assuming a
validated status. The current loop rejects malformed or unsupported probabilities
and permits one final synthesis turn only when evidence was retained. It can
abstain. A citation makes support inspectable, not automatically correct.

The mock example below is an explicitly synthetic flow test, not a model evaluation.

<!-- ML4T_ZH_BEGIN c0024-p000 -->
六月擷取的紀錄保留了實際搜尋及模型訊息。它早於新的驗收檢查；應檢視其理由和來源，而不要逕自認定它已通過驗證。現行迴圈會拒絕格式錯誤或缺乏依據的機率，且只有在保留了證據時才允許最後一輪統整。它可以棄答。引用來源讓依據可以受檢視，但不保證結論正確。

下方的模擬範例明確是一項合成流程測試，而非模型評估。
<!-- ML4T_ZH_END c0024-p000 -->


In [14]:
mock_llm = MockLLMClient()
mock_search = create_search_client("mock")

p_mock, _, mock_traces, mock_tokens = run_react_agent(mock_llm, mock_search, question, max_steps=3)

print(f"Mock provider: {mock_llm.model_name}")
print(f"Mock steps: {len(mock_traces)}")
print(f"Mock tokens: {mock_tokens.total_tokens:,}")
print(f"Synthetic p(YES): {p_mock}")

Step 0: search -> 2 eligible results
Mock provider: mock-model
Mock steps: 4
Mock tokens: 2,496
Synthetic p(YES): None


## Key Takeaways

1. **One interface, any backend.** An agent that calls `complete_with_usage` and nothing else
   can be moved between a free local model and a commercial API without an edit. Write the
   loop against the protocol, not against a vendor's SDK.
2. **A small action space is what makes a loop auditable.** With two actions, every turn is
   either a query you can re-issue or a probability you can score. Adding actions adds ways
   for a run to go wrong that the record cannot explain.
3. **Model output is untrusted input.** Parse it against the declared schema, and hand a
   malformed reply back for correction rather than letting it reach a tool.
4. **A budget needs a distinguishable exhaustion value.** A loop that returns a probability on
   both success and failure gives the caller no way to tell a judgement from a timeout. Return
   a value the caller must branch on, and say so in the type or the record.
5. **Record the conversation, not just the answer.** Prompts, replies and tool results in one
   file are what make a run reproducible after the market has closed and the model has been
   retired.

**Known limitations of what is built here.** The loop keeps every message in the context
window, so a long session eventually exceeds it and there is no summarisation or eviction. The
search results carry no publication dates, so a run cannot be shown to be free of hindsight.
There is one agent and one sample, so the probability has no dispersion around it. And nothing
scores the forecast: an agent is only as good as the record of how its past probabilities
resolved, which needs resolved questions and a proper scoring rule.


<!-- ML4T_ZH_BEGIN c0026-p000 -->
## 重點整理

1. **單一介面，適用任何後端。** 只呼叫 `complete_with_usage`、不呼叫其他介面的代理，不必修改就能在免費本機模型和商業 API 之間移轉。應依協定撰寫迴圈，而非綁定供應商 SDK。
2. **精簡的行動空間讓迴圈可稽核。** 只有兩種行動時，每一輪不是可重新發出的查詢，就是可評分的機率。增加行動，也會增加執行紀錄無法解釋的出錯方式。
3. **模型輸出是不可信任的輸入。** 依宣告的綱要解析輸出；若回覆格式錯誤，應送回模型修正，避免它傳到工具。
4. **用盡預算時需要可辨識的回傳值。** 若迴圈無論成功失敗都回傳機率，呼叫端就無法分辨判斷和逾時。應回傳讓呼叫端必須據以分支處理的值，並在型別或紀錄中說明。
5. **記錄對話，而不只是答案。** 將提示詞、回覆與工具結果存於同一檔案，才能在市場結算、模型退役後重現執行過程。

**此處實作的已知限制。** 迴圈將每則訊息都保留在上下文視窗中，因此工作階段過長終會超出上限，而且沒有摘要或移除機制。搜尋結果沒有發布日期，因此無法證明執行未使用事後才得知的資訊。只有一個代理和一次取樣，因此無法觀察機率的離散程度。此外，程式沒有為預測評分：代理的品質取決於其過往機率預測結果的紀錄，而這需要已結算問題與適當的評分規則。
<!-- ML4T_ZH_END c0026-p000 -->

**Next**: [`02_tool_contracts`](02_tool_contracts.ipynb) gives the search tool a typed
contract, a publication-date filter, and a source policy.

<!-- ML4T_ZH_BEGIN c0026-p001 -->
**下一步**：[`02_tool_contracts`](02_tool_contracts.ipynb) 會為搜尋工具加入具型別的契約、發布日期篩選器，以及來源政策。
<!-- ML4T_ZH_END c0026-p001 -->
